# Notebook 04 — Field Mapping

This notebook renders an interactive Leaflet map of Iowa field boundaries
colored by crop type, with NRCS SSURGO soil data in each popup.

**Requirements:** `folium` — install with `pip install folium`

**Data source:** `data/fields_with_soil.geojson`


In [ ]:
import json
from pathlib import Path

ROOT             = Path("..")
ENRICHED_GEOJSON = ROOT / "data" / "fields_with_soil.geojson"

with open(ENRICHED_GEOJSON) as f:
    geojson = json.load(f)

print(f"Loaded {len(geojson['features'])} features")

# Detect soil property key (supports both 'soil' and 'nrcs_soil')
SOIL_KEY = "soil" if "soil" in geojson["features"][0]["properties"] else "nrcs_soil"
print(f"Soil property key: '{SOIL_KEY}'")

In [ ]:
try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
    print("folium not installed — run: pip install folium")
    print("Falling back to plain text field summary.")

## 1  Compute map center

In [ ]:
def bbox_center(features: list[dict]) -> tuple[float, float]:
    """Return (lat, lon) center of all feature vertices."""
    lons, lats = [], []
    for feat in features:
        for ring in feat["geometry"]["coordinates"]:
            for lon, lat in ring:
                lons.append(lon)
                lats.append(lat)
    return (sum(lats) / len(lats), sum(lons) / len(lons))

center = bbox_center(geojson["features"])
print(f"Map center: lat={center[0]:.4f}, lon={center[1]:.4f}")

## 2  Build interactive map

In [ ]:
CROP_COLORS = {
    "corn":    "#f5a623",
    "soybean": "#417505",
    "wheat":   "#c8a850",
}


def style_function(feature: dict) -> dict:
    crop = feature["properties"].get("crop", "unknown")
    return {
        "fillColor":   CROP_COLORS.get(crop, "#888888"),
        "color":       "#333333",
        "weight":      1,
        "fillOpacity": 0.6,
    }


def popup_html(props: dict) -> str:
    soil = props.get(SOIL_KEY, {})
    lines = [
        f"<b>Field ID:</b> {props.get('field_id')}",
        f"<b>Crop:</b> {props.get('crop')}",
        f"<b>Region:</b> {props.get('region')}",
        f"<b>Area:</b> {props.get('area_acres')} ac",
        f"<b>Soil type:</b> {soil.get('soil_type', 'N/A')}",
        f"<b>pH:</b> {soil.get('ph', 'N/A')}",
        f"<b>Organic matter:</b> {soil.get('organic_matter', 'N/A')}%",
    ]
    if soil.get("drainage_class"):
        lines.append(f"<b>Drainage:</b> {soil['drainage_class']}")
    return "<br>".join(lines)


if HAS_FOLIUM:
    m = folium.Map(location=center, zoom_start=7, tiles="CartoDB positron")

    folium.GeoJson(
        geojson,
        name="Iowa Fields",
        style_function=style_function,
        tooltip=folium.GeoJsonTooltip(
            fields=["field_id", "crop", "region"],
            aliases=["Field ID", "Crop", "Region"],
        ),
    ).add_to(m)

    for feat in geojson["features"]:
        props = feat["properties"]
        ring  = feat["geometry"]["coordinates"][0]
        c_lon = sum(c[0] for c in ring) / len(ring)
        c_lat = sum(c[1] for c in ring) / len(ring)
        folium.Marker(
            location=[c_lat, c_lon],
            popup=folium.Popup(popup_html(props), max_width=280),
            icon=folium.DivIcon(html=""),
        ).add_to(m)

    folium.LayerControl().add_to(m)

    out_html = ROOT / "data" / "fields_with_soil_map.html"
    m.save(str(out_html))
    print(f"Map saved to {out_html}")
    display(m)  # noqa: F821
else:
    for feat in geojson["features"]:
        p    = feat["properties"]
        soil = p.get(SOIL_KEY, {})
        print(
            f"Field {p['field_id']:2d}  crop={p['crop']:7s}  "
            f"soil={soil.get('soil_type','?'):25s}  "
            f"pH={soil.get('ph','?')}  OM={soil.get('organic_matter','?')}%"
        )

## 3  Legend

In [ ]:
if HAS_FOLIUM:
    from IPython.display import HTML
    swatches = "".join(
        f'<span style="background:{color};padding:3px 10px;margin:2px;border-radius:3px;color:white">{crop}</span> '
        for crop, color in CROP_COLORS.items()
    )
    display(HTML(f"<div><b>Legend:</b> {swatches}</div>"))  # noqa: F821
else:
    for crop, color in CROP_COLORS.items():
        print(f"  {crop}: {color}")